# Sliding Window Rate Limiter

A sliding window rate limiter tracks requests within a rolling time window.
Unlike a fixed window (which resets at a set interval), a sliding window
prevents burst traffic at window boundaries.

This example shows two implementations:
- **Sorted set approach** — precise sliding window using `ZADD` / `ZREMRANGEBYSCORE`
- **Lua script approach** — atomic version safe for concurrent multi-instance deployments

## Setup

In [ ]:
import time
import uuid

import redis

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

## Sorted Set Implementation

Each request is stored as a member of a sorted set with the timestamp as score.
On every request:
1. Remove entries older than the window
2. Count remaining entries
3. If under the limit, add the new entry and allow
4. Otherwise reject

In [ ]:
def is_allowed_sorted_set(
    client: redis.Redis,
    key: str,
    limit: int,
    window_seconds: int,
) -> bool:
    """
    Returns True if the request is within the rate limit, False otherwise.

    Args:
        client: Redis client instance
        key: unique identifier for the rate limit bucket (e.g. user_id or ip)
        limit: maximum number of requests allowed in the window
        window_seconds: size of the sliding window in seconds
    """
    now = time.time()
    window_start = now - window_seconds

    pipe = client.pipeline()
    # Remove requests outside the sliding window
    pipe.zremrangebyscore(key, '-inf', window_start)
    # Count requests in the current window
    pipe.zcard(key)
    _, count = pipe.execute()

    if count < limit:
        # Add this request with current timestamp as score
        member = str(uuid.uuid4())
        client.zadd(key, {member: now})
        # Set TTL so the key expires after the window if unused
        client.expire(key, window_seconds * 2)
        return True

    return False

In [ ]:
# Example: 5 requests per 10 seconds for user 'user_123'
r.delete('rate:user_123')

for i in range(7):
    allowed = is_allowed_sorted_set(r, 'rate:user_123', limit=5, window_seconds=10)
    print(f'Request {i + 1}: {"allowed" if allowed else "rejected"}')

## Atomic Lua Script Implementation

The sorted set approach above uses a pipeline but is not fully atomic —
two concurrent processes could both read `count < limit` and both add.
A Lua script runs atomically on the Redis server, eliminating this race condition.

In [ ]:
SLIDING_WINDOW_SCRIPT = """
local key = KEYS[1]
local now = tonumber(ARGV[1])
local window = tonumber(ARGV[2])
local limit = tonumber(ARGV[3])
local member = ARGV[4]

local window_start = now - window

-- Remove entries outside the window
redis.call('ZREMRANGEBYSCORE', key, '-inf', window_start)

-- Count current entries
local count = redis.call('ZCARD', key)

if count < limit then
    redis.call('ZADD', key, now, member)
    redis.call('EXPIRE', key, window * 2)
    return 1
end

return 0
"""

sliding_window = r.register_script(SLIDING_WINDOW_SCRIPT)


def is_allowed_atomic(
    client: redis.Redis,
    key: str,
    limit: int,
    window_seconds: int,
) -> bool:
    result = sliding_window(
        keys=[key],
        args=[time.time(), window_seconds, limit, str(uuid.uuid4())],
    )
    return bool(result)

In [ ]:
# Same test using the atomic Lua version
r.delete('rate:user_456')

for i in range(7):
    allowed = is_allowed_atomic(r, 'rate:user_456', limit=5, window_seconds=10)
    print(f'Request {i + 1}: {"allowed" if allowed else "rejected"}')

## Async Version

For async applications use `redis.asyncio`:

In [ ]:
import asyncio
import redis.asyncio as aioredis


async def is_allowed_async(
    client: aioredis.Redis,
    key: str,
    limit: int,
    window_seconds: int,
) -> bool:
    now = time.time()
    window_start = now - window_seconds

    async with client.pipeline(transaction=True) as pipe:
        await pipe.zremrangebyscore(key, '-inf', window_start)
        await pipe.zcard(key)
        _, count = await pipe.execute()

    if count < limit:
        await client.zadd(key, {str(uuid.uuid4()): now})
        await client.expire(key, window_seconds * 2)
        return True

    return False


async def main():
    async_client = aioredis.Redis(host='localhost', port=6379, decode_responses=True)
    await async_client.delete('rate:user_789')

    for i in range(7):
        allowed = await is_allowed_async(
            async_client, 'rate:user_789', limit=5, window_seconds=10
        )
        print(f'Request {i + 1}: {"allowed" if allowed else "rejected"}')

    await async_client.aclose()


asyncio.run(main())

## Expected output

```
Request 1: allowed
Request 2: allowed
Request 3: allowed
Request 4: allowed
Request 5: allowed
Request 6: rejected
Request 7: rejected
```

Requests 6 and 7 are rejected because 5 requests already exist within the 10-second window.